<a href="https://colab.research.google.com/github/lalawee/special-octo-barnacle/blob/main/Task1_QLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
"""Task1_QLearning.ipynb

#OpenAI Gym - Frozen Lake (Q-Learning)
"""

import numpy as np
import gym
from IPython.display import clear_output
import time

# cause the new version of numpy has removed np.bool8 type
# and replace it with np.bool_
# gym still uses np.bool8
np.bool8 = np.bool_

#
# Parameters
#
EPISODES = 10000
GAMMA = 0.9       # discount factor
ALPHA = 0.1       # learning rate
EPS = 1.0         # initial epsilon
MIN_EPS = 0.01    # minimum epsilon
DECAY = 0.999     # epsilon decay rate per episode

In [13]:
"""## Setup Gym Env"""

#
# The reward is 1 if you reach the Goal and 0 otherwise.
# Landing on a hole will terminate the game
# is_slippery=False -> it's deterministic
# render_mode='ansi' refers to display on text-based console
#
# For details: https://gymnasium.farama.org/environments/toy_text/frozen_lake/
#
env = gym.make('FrozenLake-v1', new_step_api=True, render_mode='ansi', is_slippery=False)

print("No of states:", env.observation_space.n)
print("No of actions:", env.action_space.n)

print(env.observation_space) # meaning values are 0 to 15
print(env.action_space) # meaning values are 0 to 3

No of states: 16
No of actions: 4
Discrete(16)
Discrete(4)


In [14]:
def get_state(obs):
    """Extract integer state from whatever env.reset() returns."""
    if isinstance(obs, tuple):
        return int(obs[0])
    return int(obs)

In [15]:
# Initialize Q-table with zeros: 16 states x 4 actions
Q = np.zeros((env.observation_space.n, env.action_space.n))

total_goal = 0
epsilon = EPS

for episode in range(EPISODES):
    state = get_state(env.reset())
    truncated = False
    terminated = False

    while not (truncated or terminated):
        # Epsilon-greedy action selection
        if np.random.random() < epsilon:
            action = env.action_space.sample()  # explore
        else:
            action = np.argmax(Q[state, :])     # exploit

        result = env.step(action)
        next_state = int(result[0])
        reward = result[1]
        terminated = result[2]
        truncated = result[3]

        # Q-Learning update (off-policy): uses max over next actions
        # If terminated, there is no future value
        if terminated:
            td_target = reward
        else:
            td_target = reward + GAMMA * np.max(Q[next_state, :])

        Q[state, action] = Q[state, action] + ALPHA * (td_target - Q[state, action])

        if reward == 1:
            total_goal += 1

        state = next_state

    # Decay epsilon after each episode
    epsilon = max(MIN_EPS, epsilon * DECAY)

print(f"Training Finished. Total Goals: {total_goal}, Cumulative Success Rate: {(total_goal * 100 / EPISODES):0.1f}%")

Training Finished. Total Goals: 8810, Cumulative Success Rate: 88.1%


In [16]:
"""## Print Optimal Q-Table"""

print("Optimal Q-Table:")
print(Q)

Optimal Q-Table:
[[0.531441   0.47829681 0.59049    0.531441  ]
 [0.531441   0.         0.6561     0.59049   ]
 [0.59049    0.729      0.59048999 0.6561    ]
 [0.6561     0.         0.52222462 0.51849226]
 [0.45198333 0.38822471 0.         0.53144099]
 [0.         0.         0.         0.        ]
 [0.         0.81       0.         0.6561    ]
 [0.         0.         0.         0.        ]
 [0.1143332  0.         0.61767748 0.10649212]
 [0.24087041 0.68223746 0.80999983 0.        ]
 [0.7289982  0.9        0.         0.72899977]
 [0.         0.         0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.         0.51879614 0.89999982 0.51904547]
 [0.80999727 0.89999854 1.         0.80999921]
 [0.         0.         0.         0.        ]]


In [17]:
"""# Part 2: Visualizing Optimal Policy Movement"""

print("\nVisualizing Optimal Policy Movement:")
for episode in range(3):
    state = get_state(env.reset())
    terminated = False
    truncated = False
    step_count = 0

    while not (truncated or terminated):
        # Display the frame
        clear_output(wait=True)
        print(f"Episode {episode + 1}  |  Step {step_count}")
        for line in env.render():
            print(line)
        time.sleep(0.5)

        # Use greedy policy from learned Q-values
        action = np.argmax(Q[state, :])
        result = env.step(action)
        state = int(result[0])
        reward = result[1]
        terminated = result[2]
        truncated = result[3]
        step_count += 1

    # Show final frame
    clear_output(wait=True)
    print(f"Episode {episode + 1} - Final Result")
    for line in env.render():
        print(line)
    if reward == 1:
        print(f"=> Goal reached in {step_count} steps")
    else:
        print("=> Fell into a hole.")
    time.sleep(1.0)

Episode 3 - Final Result
  (Right)
SFFF
FHFH
FFFH
HFFG

=> Goal reached in 6 steps
